# Session 9 · Joins, binding rows and dates

Work top to bottom and run each cell with **Shift+Enter**.

Cells under a **Your turn** heading contain `...` placeholders. Replace them, run the cell, then run the check cell beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook.

The data is the **synthetic** 2025 season from Session 3 (`data/matches_2025_synthetic.csv`): invented results for practice only. Solutions are in the matching `_solutions` notebook.

**Before you start:** copy `teams.csv` from the Session 09 folder into your project's `data/` folder, next to the matches file.

In [49]:
def check(label, get, expected):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [50]:
import numpy as np
import pandas as pd

matches = pd.read_csv("../data/matches_2025_synthetic.csv", parse_dates=["date"])
matches = matches.assign(
    home_score=lambda d: d["home_goals"] * 6 + d["home_behinds"],
    away_score=lambda d: d["away_goals"] * 6 + d["away_behinds"],
    total_goals=lambda d: d["home_goals"] + d["away_goals"],
    margin=lambda d: d["home_score"] - d["away_score"],
)
matches.head(3)

,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd,home_score,away_score,total_goals,margin
0,1,2025,1,2025-03-16,Collingwood,Carlton,12,12,19,11,14893.0,84,125,31,-41
1,2,2025,1,2025-03-16,Hawthorn,Brisbane,14,9,17,10,12927.0,93,112,31,-19
2,3,2025,1,2025-03-15,North Melbourne,Richmond,16,11,10,9,21940.0,107,69,26,38


In [51]:
teams = pd.read_csv("../data/teams.csv")
teams.head()

,team,nickname,state
0,Adelaide,Crows,SA
1,Brisbane,Lions,QLD
2,Carlton,Blues,VIC
3,Collingwood,Magpies,VIC
4,Essendon,Bombers,VIC


## 1. Joins

| dplyr | pandas |
| --- | --- |
| `left_join(x, y, by = "team")` | `x.merge(y, on="team", how="left")` |
| `left_join(x, y, by = c("home_team" = "team"))` | `x.merge(y, left_on="home_team", right_on="team", how="left")` |
| `inner_join`, `full_join` | `how="inner"`, `how="outer"` |
| `anti_join(x, y)` | `x.merge(y, how="left", indicator=True)`, then keep `_merge == "left_only"` |
| suffixes `.x` / `.y` | `suffixes=("_x", "_y")`, which you can rename |

One habit worth building now: pass `validate="many_to_one"` on lookups. pandas then raises an error if the lookup table has duplicate keys, which would otherwise silently multiply your rows.

### Your turn 1.1

Left-join `teams` onto `matches` to get each **home** team's state. Call the result `home_info`, then count the rows where `state` came back missing. Which team is it?

In [52]:
home_info = matches.merge(teams, left_on="home_team", right_on="team", how="left")
n_missing_state = home_info["state"].isnull().sum()

In [53]:
check("1.1 n_missing_state", lambda: n_missing_state, 13)

PASS     1.1 n_missing_state


### Your turn 1.2: find the mismatch properly

Count-and-guess does not scale. An anti-join lists exactly which names failed to match:

1. `all_teams`: a one-column DataFrame named `team`, holding every distinct team in `matches` (home or away)
2. `unmatched`: a **sorted list** of the teams in `all_teams` that have no match in `teams`

Hint: `pd.concat([matches["home_team"], matches["away_team"]]).drop_duplicates()` gets the distinct names; `.to_frame("team")` turns them into a DataFrame.

In [54]:
# 1. `all_teams`: a one-column DataFrame named `team`, holding every distinct team in `matches` (home or away)
all_teams = pd.concat([matches["home_team"], matches["away_team"]]).drop_duplicates().reset_index(drop=True)
all_teams = all_teams.to_frame("team")
#2. `unmatched`: a **sorted list** of the teams in `all_teams` that have no match in `teams`
unmatched = sorted(set(all_teams["team"]) - set(teams["team"]))

In [55]:
check("1.2 unmatched", lambda: unmatched, ["GWS"])

PASS     1.2 unmatched


### Your turn 1.3

Fix the lookup rather than the data. Create `teams_fixed`, where "Greater Western Sydney" becomes "GWS". Then join it onto `matches` **twice**, once for the home team and once for the away team, and count the matches between teams from different states.

Hint: `.replace({"team": {"old": "new"}})` changes values in one column. On the second join, `suffixes=("_home", "_away")` keeps the two state columns apart.

In [56]:
teams_fixed = teams.replace({"team": {"Greater Western Sydney": "GWS"}})
both = matches.merge(teams_fixed, left_on="home_team", right_on="team", how="left").merge(teams_fixed, left_on="away_team", right_on="team", how="left", suffixes=("_home", "_away"))
# count the matches between teams from different states.
n_interstate = both[both["state_home"] != both["state_away"]].shape[0]

In [57]:
check("1.3 no missing states", lambda: int(both[["state_home", "state_away"]].isna().sum().sum()), 0)
check("1.3 n_interstate", lambda: n_interstate, 140)

PASS     1.3 no missing states
PASS     1.3 n_interstate


## 2. Binding rows: `bind_rows`

`pd.concat([a, b], ignore_index=True)` stacks DataFrames with matching columns, like `bind_rows(a, b)`. `ignore_index=True` gives the result a fresh 0, 1, 2... index.

The classic use in football data: turn one-row-per-**match** into one-row-per-**team-per-match**, so every ladder or team statistic becomes a simple `groupby("team")`.

### Your turn 2.1

Build `team_games` with these columns: `match_id`, `round`, `date`, `team`, `opponent`, `score_for`, `score_against`.

1. `home_rows`: rename `home_team` to `team`, `away_team` to `opponent`, `home_score` to `score_for` and `away_score` to `score_against`
2. `away_rows`: the mirror image
3. Keep only the seven columns above in each, then stack them with `pd.concat`

In [58]:
cols = ["match_id", "round", "date", "team", "opponent", "score_for", "score_against"]
home_rows = matches.rename(columns={"home_team": "team", "away_team": "opponent", "home_score": "score_for", "away_score": "score_against"})
away_rows = matches.rename(columns={"home_team": "opponent", "away_team": "team", "home_score": "score_against", "away_score": "score_for"})
team_games = pd.concat([home_rows[cols], away_rows[cols]], ignore_index=True)

In [59]:
check("2.1 rows", lambda: len(team_games), 414)
check("2.1 Geelong points for", lambda: int(team_games.loc[team_games["team"] == "Geelong", "score_for"].sum()), 1853)

PASS     2.1 rows
PASS     2.1 Geelong points for


### Your turn 2.2: the ladder

From `team_games`, build `ladder` with one row per team: `wins`, `draws`, `points_for`, `points_against`, then `points` (4 per win, 2 per draw) and `percentage` (points for ÷ points against × 100). Sort by points, then percentage, both descending, as the AFL does.

Set `top_team` and `top_points` from the first row.

In [60]:
team_games = team_games.assign(
    wins=lambda d: (d["score_for"] > d["score_against"]).astype(int),
    draws=lambda d: (d["score_for"] == d["score_against"]).astype(int),
    points=lambda d: 4 * d["wins"] + 2 * d["draws"],
)

ladder = team_games.groupby("team").agg(
    score_for = ("score_for", "sum"),
    score_against = ("score_against", "sum"),
    wins = ("wins", "sum"),
    draws = ("draws", "sum"),
    points = ("points", "sum"),
).reset_index()

ladder["percentage"] = (ladder["score_for"] / ladder["score_against"] * 100).round(2)
ladder = ladder.sort_values(["points", "percentage"], ascending=False)

top_team = ladder.iloc[0]["team"]
top_points = ladder.iloc[0]["points"]

In [61]:
check("2.2 top_team", lambda: top_team, 'Fremantle')
check("2.2 top_points", lambda: top_points, 72)

PASS     2.2 top_team
PASS     2.2 top_points


## 3. Dates

`parse_dates=["date"]` already turned the column into real dates. The `.dt` accessor then works like lubridate:

| lubridate | pandas |
| --- | --- |
| `month(date)` | `s.dt.month` |
| `wday(date, label = TRUE)` | `s.dt.day_name()` |
| `floor_date(date, "month")` | `s.dt.to_period("M")` |
| summarise by calendar month | `df.resample("MS", on="date")["x"].sum()` (`"MS"` = month start) |

In [62]:
matches["date"].dt.day_name().value_counts()

date
Saturday    81
Thursday    43
Friday      42
Sunday      41
Name: count, dtype: int64

### Your turn 3.1

- `n_april`: the number of matches played in April
- `busiest_day`: the day of the week with the most matches, as a string (`.value_counts().idxmax()`)
- `crowd_by_month`: total crowd per calendar month, using `resample`

In [63]:
n_april = matches[matches["date"].dt.month == 4].shape[0]
busiest_day = matches["date"].dt.day_name().value_counts().idxmax()
# total crowd for each month using resample
crowd_by_month = matches.resample("MS", on="date")["crowd"].sum()

In [64]:
check("3.1 n_april", lambda: n_april, 36)
check("3.1 busiest_day", lambda: busiest_day, 'Saturday')
check("3.1 April crowd", lambda: float(crowd_by_month.loc["2025-04-01"]), 1408728.0)

PASS     3.1 n_april
PASS     3.1 busiest_day
PASS     3.1 April crowd


## Wrap-up

Run All, confirm every check passes, and commit from VS Code (message: `Session 9: joins, binding rows and dates`). Then spend the rest of the session on the Milestone 2 decision in the Session 9 guide.